# YAML Sandbox

Sandbox notebook to learn how to use YAML for DEMONEXT.

Applications:
 * runtime configuration files
 * observation definition files

YAML (YAML Ain't Markup Language) is a simple, structured, ascii-based data serialization language used
across many  languages and operating systems that is often used for human-readable and editable runtime
configuration files because of its simple syntax.

### Sources

 * Official YAML website: https://yaml.org/
 * PyYAML, the standard python YAML loader/emitter (comes with Anaconda): http://pyyaml.org 
 * A nice YAML tutorial: https://www.cloudbees.com/blog/yaml-tutorial-everything-you-need-get-started
 * Another YAML tutorial: https://python.land/data-processing/python-yaml


In [2]:
import os

# YAML module

import yaml

## Load and parse a YAML file

### loadYAML() function

We use the yaml `safe_load()` method instead of the deprecated `load()` method.

In [3]:
def loadYAML(fileName):
    """
    Load the contents of a YAML-format file using safe_load()
    """
    
    if os.path.exists(fileName):
        try:
            with open(fileName,'r') as stream:
            try:
                data = yaml.safe_load(stream)
            except yaml.YAMLError as exp:
                raise RuntimeError(f"ERROR: could not load {fileName}: {exp}")
        except Exception as err:
            raise RuntimError(f"YAML file {fileName} not found.")
            
        if len(data)==0:
            raise RuntimeError(f"YAML file {fileName} is empty!")
        
        return data
    else:
        raise ValueError(f"YAML file {fileName} does not exist")        

### read and print out

Note that pyYAML strips out all comments (block and inline), so this is a formatted printout of the
file contents without the comments.

In [4]:
myFile = "demonext.txt"

try:
    data = loadYAML(myFile)
    print(f"\nRead YAML file {myFile}, contents:")
    for key in data:
        print(f"\n{key}:")
        yamlItem = data[key]
        for keyword in yamlItem:
            print(f"  {keyword}: {yamlItem[keyword]}")
except Exception as exp:
    print(f"ERROR: loadYAML() - {exp}")



Read YAML file demonext.txt, contents:

directories:
  DataDir: D:\Data
  LogDir: D:\Logs

site:
  OBSERVAT: DEMONEXT Reboot2025
  SITENAME: Sierra Remote Observatories
  SITE_LON: -119.41293
  SITE_LAT: 37.07031
  SITE_EL: 1405.0

instrument:
  TELESCOP: PlaneWave CDK20 f/6.8
  INSTRUME: FLI PL23042 + CFW3-10
  FOCALLEN: 3454.0
  APTDIA: 0.508
  APTAREA: 171854.9
  FOCUSER: PlaneWave Hedrick
  FOCUSSSZ: 1.0

pdu:
  config: .demonext/config/raritanPDU.txt

ccd:
  Setpoint: -20
  MaxExpTime: 1800.0
  Binning: [1, 1]
  FWStepTime: 1.2

guider:
  TELESCOP: SkyWatcher 102mm f/12.7
  INSTRUME: ZWO ASI421mm
  FILTER: Yellow Wratten 12
  Aggressiveness: 7
  ExpTimes: [1, 2, 5, 10, 20, 30, 60]

focuser:
  MaxFocus: 33000
  MaxExpTime: 120.0
  RefFocus: 1000
  RefFilter: 0
  FilterOffsets: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

calibration:
  PROJECT: Calibration
  PI_NAME: Calibration
  PI_INST: All
  OBJECT: Calibration

project:
  PROJECT: Engineering
  PI_NAME: Engineering
  PI_INST: OSU
  OBJECT

## Create content and "emit" into a YAML file

Use the `yaml.dump()` method to create a new yaml file.  We can also append to an existing file.

Note that when a python dictionary is emitted, the order of keywords in each subdictionary block is 
alphabetized by default unless you use the `sort_keys=False` argument.

In [5]:
outFile = "obsTest.obs"

# multi-item dictionary

newDict = {}

# some dictionaries to emit into a YAML file

projInfo = {"PROJECT":"OSU_Galaxies",
            "PI_NAME":"R. Pogge",
            "PI_INST":"OSU",
            "PI_EMAIL":"pogge.1@osu.edu"}

newDict["project"] = projInfo

# target information

targInfo = {"Name":"NGC1234",
            "RA2000":"12:13:14.567",
            "Dec2000":"+22:33:44.12"}

newDict["target"] = targInfo

# observation sequence: objN = ["filterID",expTime,numExp]

numRepeat = 0 # number of times to repeat the sequence (0 means no repeats, same as Repeat: 1)

# guiding modes:
#     auto - autoguider telescope
#  science - science-image guiding
#     none - no guiding, run open loop

guideMode = "none"

obs = []
obs.append(["g_SDSS",300,3])
obs.append(["r_SDSS",300,3])
obs.append(["i_SDSS",360,3])

obsInfo = {}

obsInfo["GuideMode"] = guideMode

numObs = len(obs)
obsInfo["NumObs"] = numObs

if numRepeat > 0:
    obsInfo["Repeat"] = numRepeat
    
for i in range(numObs):
    key = f"obs{i+1}"
    obsInfo[key] = obs[i]

newDict["sequence"] = obsInfo

# constraints

conInfo = {}
conInfo["Airmass"] = 1.5  # maximum airmass
conInfo["MoonPhase"] = 0.3 # maximum moon illumination, 0.3 is dark, 0.6 gray, 1.0 bright
conInfo["MoonAngle"] = 30.0 # minimum moon angle in degrees
conInfo["MaxSky"] = 25000.0 # maximum sky brightness in adu/pixel

# optional: start/end for time-sensitive observations
#
# must be explicity UTC ISO8601-compliant UTC date/time format:  2025-01-29T00:12:13Z 
#
conInfo["StartTime"] = "2025-01-29T00:12:13Z" # means "do not start before"
conInfo["EndTime"] = "2025-01-29T04:13:14Z"   # means "do not observe after"
#

newDict["constraints"] = conInfo

# create the new yaml file

with open(outFile, "w") as file:
    yaml.dump(newDict,file,sort_keys=False)
    file.close()

### what does it look like when read back in?


In [6]:
obsFile = "obsTest.obs"

try:
    obs = loadYAML(obsFile)
    print(f"\nRead observation file {obsFile}, contents:")
    for key in obs:
        print(f"\n{key}:")
        yamlItem = obs[key]
        for keyword in yamlItem:
            print(f"  {keyword}: {yamlItem[keyword]}")
except Exception as exp:
    print(f"ERROR: loadConfig() - {exp}")



Read observation file obsTest.obs, contents:

project:
  PROJECT: OSU_Galaxies
  PI_NAME: R. Pogge
  PI_INST: OSU
  PI_EMAIL: pogge.1@osu.edu

target:
  Name: NGC1234
  RA2000: 12:13:14.567
  Dec2000: +22:33:44.12

sequence:
  GuideMode: none
  NumObs: 3
  obs1: ['g_SDSS', 300, 3]
  obs2: ['r_SDSS', 300, 3]
  obs3: ['i_SDSS', 360, 3]

constraints:
  Airmass: 1.5
  MoonPhase: 0.3
  MoonAngle: 30.0
  MaxSky: 25000.0
  StartTime: 2025-01-29T00:12:13Z
  EndTime: 2025-01-29T04:13:14Z


In [7]:
# try something

obsFile = 'myObs.obs'

try:
    obs = loadYAML(obsFile)

    projInfo = obs['project']
    obsInfo = obs['sequence']
    targInfo = obs['target']
    numObs = obsInfo['NumObs']
    totExec = 0
    print(f"Observations of {targInfo['Name']} ({targInfo['RA2000']} {targInfo['Dec2000']})")
    print(f"\nImage Sequence:")
    for i in range(numObs):
        key = f"obs{i+1}"
        filt = obsInfo[key][0]
        expT = obsInfo[key][1]
        numExp = obsInfo[key][2]
        print(f"  Obs{i+1}: {filt} filter, {numExp}x{expT:.1f} sec")
    if "Repeat" in obsInfo:
        numReps = obsInfo['Repeat']
        print(f"  Repeat sequence {numReps} times (execute {numReps+1} sequences)")
    if "GuideMode" in obsInfo:
        print(f"  GuideMode: {obsInfo['GuideMode']}")

    if "constraints" in obs:
        conInfo = obs["constraints"]
        print("\nConstraints:")
        if "Airmass" in conInfo:
            print(f"  Airmass < {conInfo['Airmass']:.1f}")
        else:
            print("  No airmass constraint")
        if "MoonPhase" in conInfo:
            print(f"  MoonPhase < {conInfo['MoonPhase']:1f} illuminated")
        else:
            print("  No moon phase constraint")
        if "MoonAngle" in conInfo:
            print(f"  Moon Angle < {conInfo['MoonAngle']:.1f} degrees")
        else:
            print("  No moon angle constraint")
        if "MaxSky" in conInfo:
            print(f"  Sky Counts < {conInfo['MaxSky']:.1f} adu")
        else:
            print("  No background constraint")
        if "StartTime" in conInfo:
            print(f"  Start after {conInfo['StartTime']}")
        else:
            print(f"  No starting time constraint")
        if "EndTime" in conInfo:
            print(f"  End before {conInfo['EndTime']}")
        else:
            print(f"  No ending time constraint")
    else:
        print("No observing constraints")
        
except Exception as exp:
    print(f"ERROR: {exp}")

Observations of NGC1234/35 Group (12:13:14.567 +22:33:44.12)

Image Sequence:
  Obs1: g_SDSS filter, 3x90.0 sec
  Obs2: r_SDSS filter, 3x60.0 sec
  Obs3: i_SDSS filter, 3x120.0 sec
  Repeat sequence 2 times (execute 3 sequences)
  GuideMode: none

Constraints:
  Airmass < 1.5
  MoonPhase < 0.500000 illuminated
  Moon Angle < 30.0 degrees
  Sky Counts < 25000.0 adu
  Start after 2025-01-26T02:04:05Z
  End before 2025-01-26T05:45:00Z
